# Cluster analysis: lab notebook

**Statistical Modelling and Forecasting, Multivariate and advanced methods**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist

rng = np.random.default_rng(12)

### 1. Thirty-six monthly series of three types: trend, seasonal, noise only

In [ ]:
# 1. Thirty-six monthly series of three types: trend, seasonal, noise only
t = np.arange(48)
def make(kind):
    y = 0.4 * rng.standard_normal(48) * 2
    if kind == 0: y += 0.12 * t
    if kind == 1: y += 2.2 * np.sin(2 * np.pi * t / 12)
    return y
truth = np.repeat([0, 1, 2], 12)
S = np.array([make(k) for k in truth])

### 2. Two features per series: trend R2 and seasonal R2 (month dummies on the detrended series)

In [ ]:
# 2. Two features per series: trend R2 and seasonal R2 (month dummies on the detrended series)
def r2(X, y):
    b = np.linalg.lstsq(X, y, rcond=None)[0]; e = y - X @ b
    return 1 - e @ e / ((y - y.mean()) @ (y - y.mean()))
Xt = np.column_stack([np.ones(48), t])
D = np.column_stack([np.ones(48)] + [(t % 12 == m).astype(float) for m in range(1, 12)])
feat = []
for y in S:
    res = y - Xt @ np.linalg.lstsq(Xt, y, rcond=None)[0]
    feat.append([r2(Xt, y), r2(D, res)])
feat = np.array(feat)
print("mean features by type (trend R2, seasonal R2):\n", pd.DataFrame(feat).groupby(truth).mean().round(2))

### 3. k-means with several starts

In [ ]:
# 3. k-means with several starts
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(feat)
print("\nk-means: sizes", np.bincount(km.labels_), "| adjusted Rand index vs truth:", round(adjusted_rand_score(truth, km.labels_), 3), "| WSS:", round(km.inertia_, 3))

### 4. Choosing k: elbow and silhouette

In [ ]:
# 4. Choosing k: elbow and silhouette
print("\nk   WSS     silhouette")
for k in range(1, 9):
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(feat)
    sil = silhouette_score(feat, m.labels_) if k > 1 else float("nan")
    print(f"{k}  {m.inertia_:6.3f}  {sil:.3f}")

### 5. Hierarchical clustering: linkage matters

In [ ]:
# 5. Hierarchical clustering: linkage matters
for method in ("average", "complete", "single", "ward"):
    lab = AgglomerativeClustering(n_clusters=3, linkage=method).fit_predict(feat)
    print(f"{method:9s}: sizes {np.bincount(lab)}, ARI {adjusted_rand_score(truth, lab):.3f}")

### 6. Clustering the raw series instead: correlation distance on the standardised series

In [ ]:
# 6. Clustering the raw series instead: correlation distance on the standardised series
Z = StandardScaler().fit_transform(S.T).T
L = linkage(pdist(Z, metric="correlation"), method="average")
lab = fcluster(L, 3, criterion="maxclust") - 1
print("\ncorrelation distance on raw series: ARI", round(adjusted_rand_score(truth, lab), 3), "(captures shape, not level)")

### 7. One k-means iteration by hand (the numerical problem)

In [ ]:
# 7. One k-means iteration by hand (the numerical problem)
P = np.array([[2, 9], [3, 2], [9, 10], [1, 10], [1, 3], [2, 9.5]])
c1, c2 = P[1], P[2]
a = np.linalg.norm(P - c1, axis=1) < np.linalg.norm(P - c2, axis=1)
m1, m2 = P[a].mean(0), P[~a].mean(0)
print("\nassigned to centre 1:", np.where(a)[0] + 1, "| new centres", m1.round(2), m2.round(2), "| WCSS", round(((P[a] - m1) ** 2).sum() + ((P[~a] - m2) ** 2).sum(), 3))

## Expected output

Running all cells above should print:

```
mean features by type (trend R2, seasonal R2):
       0     1
0  0.82  0.23
1  0.04  0.80
2  0.02  0.20

k-means: sizes [12 12 12] | adjusted Rand index vs truth: 1.0 | WSS: 0.17

k   WSS     silhouette
1   7.994  nan
2   2.320  0.698
3   0.170  0.868
4   0.116  0.776
5   0.068  0.673
6   0.054  0.503
7   0.041  0.523
8   0.034  0.524
average  : sizes [12 12 12], ARI 1.000
complete : sizes [12 12 12], ARI 1.000
single   : sizes [12 12 12], ARI 1.000
ward     : sizes [12 12 12], ARI 1.000

correlation distance on raw series: ARI 0.465 (captures shape, not level)

assigned to centre 1: [2 5] | new centres [2.  2.5] [3.5  9.62] | WCSS 44.188
```

## Practice exercises

Try these in the cells below.

1. Use k-means with k = 3 on the three-group data. Compare the elbow with the silhouette.
2. Switch to single linkage with noisy data and look for chaining.
3. Choose k = 2 and k = 5 and describe what each solution does with the true groups.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
